In [1]:
# ==============================================================================
# BLOQUE 5 - CARGA RÁPIDA DEL MODELO YA ENTRENADO (aplicación directa)
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

ruta_modelo_guardado = "/content/drive/MyDrive/Proyecto_NLP/modelo_estres_final"

print("Cargando modelo y tokenizer ya entrenados...")
modelo_cargado = AutoModelForSequenceClassification.from_pretrained(ruta_modelo_guardado)
tokenizer_cargado = AutoTokenizer.from_pretrained(ruta_modelo_guardado)

# Mover a GPU si está disponible
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo_cargado.to(dispositivo)
modelo_cargado.eval()

print(f"✅ Modelo cargado y listo en: {dispositivo}")

# ------------------------------------------------------------------
# FUNCIÓN DE PREDICCIÓN
# ------------------------------------------------------------------
def predecir_estres(texto):
    inputs = tokenizer_cargado(texto, return_tensors="pt", truncation=True,
                                 padding="max_length", max_length=96).to(dispositivo)
    with torch.no_grad():
        outputs = modelo_cargado(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    confianza = probs[0][pred].item()
    etiqueta = "Estrés (1)" if pred == 1 else "No Estrés (0)"
    return etiqueta, round(confianza, 4)

# ------------------------------------------------------------------
# PRUEBA RÁPIDA EN VIVO
# ------------------------------------------------------------------
ejemplos_prueba = [
    "Estoy súper agobiado con los tres exámenes de esta semana, no doy más.",
    "ese prof, esta en na, puro lee ppt, yo miento"
]

print("\n--- PRUEBA EN VIVO ---")
for texto in ejemplos_prueba:
    etiqueta, confianza = predecir_estres(texto)
    print(f"\nTexto: {texto}")
    print(f"Predicción: {etiqueta} (confianza: {confianza})")

Mounted at /content/drive
Cargando modelo y tokenizer ya entrenados...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Modelo cargado y listo en: cuda

--- PRUEBA EN VIVO ---

Texto: Estoy súper agobiado con los tres exámenes de esta semana, no doy más.
Predicción: Estrés (1) (confianza: 0.9883)

Texto: ese prof, esta en na, puro lee ppt, yo miento
Predicción: Estrés (1) (confianza: 0.7347)
